# Phase 3: Tiled Inference (SAHI) & Out-of-Distribution (OOD) Testing
### Capstone Project: High-Resolution Full-Board Inspection & Generalization
**Hardware:** Windows 10/11 | NVIDIA RTX 4090 (24GB VRAM)  
**Tools:** SAHI (Slicing Aided Hyper Inference), OpenCV  
**Goals:**
1. Implement tiled slicing inference to inspect un-cropped full board images (e.g., 2777x2138 px) without downscaling loss.
2. Optimize slice sizes, overlaps, and NMS/NMM box merging.
3. Test out-of-distribution (OOD) generalization against synthetic-to-real gap (e.g., DeepPCB or Mendeley PCB datasets).


## 1. Install & Verify SAHI


In [ ]:
# Ensure sahi is installed:
# !pip install sahi

import sahi
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
print(f"SAHI Version: {sahi.__version__}")


## 2. Load Phase 1/2 Champion Model into SAHI
Wrap the best YOLO or RT-DETR checkpoint inside SAHI's detection abstraction.


In [ ]:
from pathlib import Path

# Select champion weights
model_path = Path("Phase1_YOLO_Baselines/YOLO26s_Phase1/weights/best.pt")
if not model_path.exists():
    model_path = Path("PCB_Capstone_Benchmark/YOLO11_Control/weights/best.pt")

print(f"Using Champion Model: {model_path}")

detection_model = AutoDetectionModel.from_pretrained(
    model_type="yolov8", # Compatible with Ultralytics YOLO models
    model_path=str(model_path),
    confidence_threshold=0.35,
    device="cuda:0"
)
print("[+] SAHI AutoDetectionModel initialized on CUDA.")


## 3. High-Resolution Full Board Sliced Inference
Simulate a full-resolution factory PCB camera frame (e.g., 2777x2138 px) and run sliced prediction.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Generate a high-resolution canvas (simulating un-cropped 2400x2400 full board)
canvas_h, canvas_w = 2400, 2400
dummy_board = np.ones((canvas_h, canvas_w, 3), dtype=np.uint8) * 45 # Dark circuit substrate

# Run SAHI sliced inference with 640x640 windows and 20% overlap
result = get_sliced_prediction(
    dummy_board,
    detection_model,
    slice_height=640,
    slice_width=640,
    overlap_height_ratio=0.2,
    overlap_width_ratio=0.2,
    postprocess_type="NMM", # Non-Maximum Merging handles boundary duplicates
    postprocess_match_metric="IOS",
    postprocess_match_threshold=0.5
)

print(f"[+] Sliced inference complete.")
print(f"Total sliced predictions found: {len(result.object_prediction_list)}")


## 4. Tiling Latency and Overlap Optimization Benchmark
Evaluate how tile dimensions and overlap ratios impact inspection speed on the RTX 4090.


In [ ]:
import time
import pandas as pd

tile_configs = [
    {"slice": 640, "overlap": 0.15},
    {"slice": 640, "overlap": 0.25},
    {"slice": 800, "overlap": 0.20},
    {"slice": 1024, "overlap": 0.20},
]

timing_results = []
test_img = np.zeros((2000, 2000, 3), dtype=np.uint8)

for cfg in tile_configs:
    t0 = time.time()
    for _ in range(5):
        _ = get_sliced_prediction(
            test_img,
            detection_model,
            slice_height=cfg["slice"],
            slice_width=cfg["slice"],
            overlap_height_ratio=cfg["overlap"],
            overlap_width_ratio=cfg["overlap"],
            verbose=0
        )
    elapsed = (time.time() - t0) / 5
    timing_results.append({
        "Slice Size": f"{cfg['slice']}x{cfg['slice']}",
        "Overlap Ratio": f"{cfg['overlap']*100:.0f}%",
        "Latency / Board (ms)": round(elapsed * 1000, 2),
        "Estimated Boards/Min": round(60.0 / elapsed, 1)
    })

df_tiling = pd.DataFrame(timing_results)
display(df_tiling)


## 5. Out-of-Distribution (OOD) Evaluation (Synthetic-to-Real Gap)
Test the trained detector on an external dataset (e.g. DeepPCB or Mendeley real PCB dataset) without fine-tuning to measure real-world domain transfer degradation.


In [ ]:
# Function to evaluate zero-shot transfer on an external test set
def evaluate_ood(model, ood_yaml_path):
    print(f"[*] Running OOD Evaluation on: {ood_yaml_path}")
    if Path(ood_yaml_path).exists():
        metrics = model.val(data=ood_yaml_path, split="test", imgsz=640)
        print(f"[+] OOD mAP@50: {metrics.box.map50:.4f}")
        print(f"[+] OOD mAP@50-95: {metrics.box.map:.4f}")
    else:
        print(f"[!] OOD dataset not found at {ood_yaml_path}. Skipping OOD execution.")

# Example path for Mendeley or DeepPCB yaml:
evaluate_ood(detection_model.model, "dataset_deeppcb.yaml")
